# Project 1: Multi-Horizon Stock Market Forecasting
## Starter Notebook - Data Pipeline & Baseline Models

**Project Goal:** Build an end-to-end forecasting system comparing traditional statistical methods with deep learning approaches

**What You'll Build:**
- Data ingestion pipeline for Indian stocks (NSE/BSE)
- Feature engineering with technical indicators
- Baseline models: ARIMA, Exponential Smoothing
- Advanced models: LSTM, GRU, Temporal Fusion Transformer
- Attention visualization and model interpretability
- Backtesting framework with risk metrics

**Timeline:** Months 1-2 of your project plan

## Setup & Installation

In [ ]:
# Install required packages
# Run this once in your terminal or uncomment below:

# !pip install yfinance pandas numpy matplotlib seaborn
# !pip install statsmodels pmdarima scikit-learn
# !pip install ta-lib-python  # For technical indicators
# !pip install pytorch-forecasting  # For Temporal Fusion Transformer
# !pip install optuna  # For hyperparameter tuning

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported successfully!")

## Phase 1: Data Collection & Exploration

We'll start with Indian stocks from NSE. Yahoo Finance uses `.NS` suffix for NSE stocks.

In [ ]:
# Configuration
STOCKS = [
    'RELIANCE.NS',  # Reliance Industries
    'TCS.NS',       # Tata Consultancy Services
    'HDFCBANK.NS',  # HDFC Bank
    'INFY.NS',      # Infosys
    'ICICIBANK.NS'  # ICICI Bank
]

START_DATE = '2018-01-01'
END_DATE = datetime.now().strftime('%Y-%m-%d')

print(f"Fetching data from {START_DATE} to {END_DATE}")
print(f"Stocks: {', '.join([s.replace('.NS', '') for s in STOCKS])}")

In [ ]:
def fetch_stock_data(ticker, start, end):
    """
    Fetch stock data from Yahoo Finance
    
    Args:
        ticker: Stock symbol (e.g., 'RELIANCE.NS')
        start: Start date (YYYY-MM-DD)
        end: End date (YYYY-MM-DD)
    
    Returns:
        DataFrame with OHLCV data
    """
    try:
        data = yf.download(ticker, start=start, end=end, progress=False)
        data['Ticker'] = ticker
        return data
    except Exception as e:
        print(f"Error fetching {ticker}: {e}")
        return None

# Fetch data for all stocks
stock_data = {}
for ticker in STOCKS:
    print(f"Downloading {ticker}...")
    data = fetch_stock_data(ticker, START_DATE, END_DATE)
    if data is not None:
        stock_data[ticker] = data
        print(f"  → {len(data)} rows fetched")

print(f"\nTotal stocks collected: {len(stock_data)}")

In [ ]:
# Let's focus on one stock for detailed analysis (you can change this)
FOCUS_STOCK = 'RELIANCE.NS'
df = stock_data[FOCUS_STOCK].copy()

print(f"\n=== {FOCUS_STOCK} Overview ===")
print(f"Date Range: {df.index[0]} to {df.index[-1]}")
print(f"Total Trading Days: {len(df)}")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nData Info:")
print(df.info())
print(f"\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# Visualize the stock price
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Close price
axes[0].plot(df.index, df['Close'], linewidth=1.5, color='#2E86AB')
axes[0].set_title(f'{FOCUS_STOCK} - Closing Price Over Time', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Price (INR)', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Volume
axes[1].bar(df.index, df['Volume'], color='#A23B72', alpha=0.6)
axes[1].set_title('Trading Volume', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Volume', fontsize=12)
axes[1].set_xlabel('Date', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Phase 2: Feature Engineering

Creating technical indicators and derived features

In [ ]:
def add_technical_indicators(df):
    """
    Add technical indicators as features
    
    Features added:
    - Returns (daily, weekly)
    - Moving Averages (SMA, EMA)
    - Volatility (rolling std)
    - RSI (Relative Strength Index)
    - MACD (Moving Average Convergence Divergence)
    - Bollinger Bands
    """
    data = df.copy()
    
    # Returns
    data['Returns'] = data['Close'].pct_change()
    data['Log_Returns'] = np.log(data['Close'] / data['Close'].shift(1))
    
    # Moving Averages
    data['SMA_5'] = data['Close'].rolling(window=5).mean()
    data['SMA_20'] = data['Close'].rolling(window=20).mean()
    data['SMA_50'] = data['Close'].rolling(window=50).mean()
    data['EMA_12'] = data['Close'].ewm(span=12, adjust=False).mean()
    data['EMA_26'] = data['Close'].ewm(span=26, adjust=False).mean()
    
    # Volatility
    data['Volatility_20'] = data['Returns'].rolling(window=20).std()
    
    # RSI
    delta = data['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    data['RSI'] = 100 - (100 / (1 + rs))
    
    # MACD
    data['MACD'] = data['EMA_12'] - data['EMA_26']
    data['MACD_Signal'] = data['MACD'].ewm(span=9, adjust=False).mean()
    data['MACD_Histogram'] = data['MACD'] - data['MACD_Signal']
    
    # Bollinger Bands
    data['BB_Middle'] = data['Close'].rolling(window=20).mean()
    bb_std = data['Close'].rolling(window=20).std()
    data['BB_Upper'] = data['BB_Middle'] + (bb_std * 2)
    data['BB_Lower'] = data['BB_Middle'] - (bb_std * 2)
    data['BB_Width'] = data['BB_Upper'] - data['BB_Lower']
    
    # Lag features
    for lag in [1, 2, 3, 5, 10]:
        data[f'Close_Lag_{lag}'] = data['Close'].shift(lag)
        data[f'Returns_Lag_{lag}'] = data['Returns'].shift(lag)
    
    # Rolling statistics
    data['Close_Rolling_Mean_7'] = data['Close'].rolling(window=7).mean()
    data['Close_Rolling_Std_7'] = data['Close'].rolling(window=7).std()
    
    # Time-based features
    data['Day_of_Week'] = data.index.dayofweek
    data['Month'] = data.index.month
    data['Quarter'] = data.index.quarter
    
    return data

# Add features
df_features = add_technical_indicators(df)

print("Technical indicators added!")
print(f"Total features: {len(df_features.columns)}")
print(f"\nFeature columns:")
print(df_features.columns.tolist())

In [ ]:
# Visualize some key indicators
fig, axes = plt.subplots(3, 1, figsize=(15, 12))

# Price with Moving Averages
axes[0].plot(df_features.index, df_features['Close'], label='Close', linewidth=1.5, color='black')
axes[0].plot(df_features.index, df_features['SMA_20'], label='SMA 20', linewidth=1.5, color='blue')
axes[0].plot(df_features.index, df_features['SMA_50'], label='SMA 50', linewidth=1.5, color='red')
axes[0].set_title('Price with Moving Averages', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# RSI
axes[1].plot(df_features.index, df_features['RSI'], linewidth=1.5, color='purple')
axes[1].axhline(y=70, color='r', linestyle='--', label='Overbought (70)')
axes[1].axhline(y=30, color='g', linestyle='--', label='Oversold (30)')
axes[1].set_title('Relative Strength Index (RSI)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('RSI')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# MACD
axes[2].plot(df_features.index, df_features['MACD'], label='MACD', linewidth=1.5, color='blue')
axes[2].plot(df_features.index, df_features['MACD_Signal'], label='Signal', linewidth=1.5, color='red')
axes[2].bar(df_features.index, df_features['MACD_Histogram'], label='Histogram', alpha=0.3, color='gray')
axes[2].set_title('MACD', fontsize=14, fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Phase 3: Stationarity Analysis

Time series forecasting requires stationary data. We'll test and transform if needed.

In [ ]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

def test_stationarity(timeseries, title=''):
    """
    Perform Augmented Dickey-Fuller test
    
    Null Hypothesis (H0): Series is non-stationary
    If p-value < 0.05, reject H0 (series is stationary)
    """
    print(f'\n=== Stationarity Test: {title} ===')
    
    # Perform ADF test
    result = adfuller(timeseries.dropna(), autolag='AIC')
    
    print(f'ADF Statistic: {result[0]:.6f}')
    print(f'p-value: {result[1]:.6f}')
    print(f'Critical Values:')
    for key, value in result[4].items():
        print(f'  {key}: {value:.3f}')
    
    if result[1] <= 0.05:
        print("✓ Series is STATIONARY (reject H0)")
    else:
        print("✗ Series is NON-STATIONARY (fail to reject H0)")
    
    return result[1] <= 0.05

# Test original close price
is_stationary_close = test_stationarity(df_features['Close'], 'Close Price')

# Test returns (typically stationary)
is_stationary_returns = test_stationarity(df_features['Returns'], 'Returns')

# Test log returns
is_stationary_log_returns = test_stationarity(df_features['Log_Returns'], 'Log Returns')

In [ ]:
# Visualize ACF and PACF for returns
fig, axes = plt.subplots(2, 1, figsize=(15, 8))

plot_acf(df_features['Returns'].dropna(), lags=40, ax=axes[0])
axes[0].set_title('Autocorrelation Function (ACF) - Returns', fontsize=14, fontweight='bold')

plot_pacf(df_features['Returns'].dropna(), lags=40, ax=axes[1])
axes[1].set_title('Partial Autocorrelation Function (PACF) - Returns', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## Phase 4: Train-Test Split

For time series, we use chronological splitting (no random shuffle)

In [ ]:
# Remove NaN values created by technical indicators
df_clean = df_features.dropna()

# Split: 80% train, 20% test
split_idx = int(len(df_clean) * 0.8)

train_data = df_clean.iloc[:split_idx]
test_data = df_clean.iloc[split_idx:]

print(f"Dataset split:")
print(f"  Training set: {len(train_data)} samples ({train_data.index[0]} to {train_data.index[-1]})")
print(f"  Test set: {len(test_data)} samples ({test_data.index[0]} to {test_data.index[-1]})")

# Visualize the split
plt.figure(figsize=(15, 6))
plt.plot(train_data.index, train_data['Close'], label='Training Data', color='blue')
plt.plot(test_data.index, test_data['Close'], label='Test Data', color='orange')
plt.axvline(x=train_data.index[-1], color='red', linestyle='--', label='Train/Test Split')
plt.title('Train-Test Split', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Close Price (INR)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Phase 5: Baseline Model - ARIMA

Auto ARIMA to find optimal (p,d,q) parameters

In [ ]:
from pmdarima import auto_arima
from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluate_forecast(y_true, y_pred, model_name='Model'):
    """
    Calculate forecasting metrics
    """
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    
    print(f"\n=== {model_name} Performance ===")
    print(f"MAE:  {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAPE: {mape:.4f}%")
    
    return {'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

# Find best ARIMA parameters
print("Finding optimal ARIMA parameters... (this may take a few minutes)")

arima_model = auto_arima(
    train_data['Close'],
    start_p=0, start_q=0,
    max_p=5, max_q=5,
    seasonal=False,
    d=None,  # Let auto_arima determine differencing
    trace=True,
    error_action='ignore',
    suppress_warnings=True,
    stepwise=True
)

print("\nBest ARIMA Model:")
print(arima_model.summary())

In [ ]:
# Make predictions on test set
n_periods = len(test_data)
arima_forecast = arima_model.predict(n_periods=n_periods)

# Evaluate
arima_metrics = evaluate_forecast(test_data['Close'].values, arima_forecast, 'ARIMA')

# Visualize predictions
plt.figure(figsize=(15, 6))
plt.plot(train_data.index, train_data['Close'], label='Training Data', color='blue', alpha=0.6)
plt.plot(test_data.index, test_data['Close'], label='Actual Test Data', color='green', linewidth=2)
plt.plot(test_data.index, arima_forecast, label='ARIMA Forecast', color='red', linewidth=2, linestyle='--')
plt.title('ARIMA Forecast vs Actual', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Close Price (INR)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Phase 6: Baseline Model - Exponential Smoothing

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Fit Exponential Smoothing model
exp_smooth_model = ExponentialSmoothing(
    train_data['Close'],
    trend='add',
    seasonal=None,  # No seasonality for daily stock data
    damped_trend=True
).fit()

# Forecast
exp_smooth_forecast = exp_smooth_model.forecast(steps=n_periods)

# Evaluate
exp_smooth_metrics = evaluate_forecast(test_data['Close'].values, exp_smooth_forecast, 'Exponential Smoothing')

# Visualize
plt.figure(figsize=(15, 6))
plt.plot(train_data.index, train_data['Close'], label='Training Data', color='blue', alpha=0.6)
plt.plot(test_data.index, test_data['Close'], label='Actual Test Data', color='green', linewidth=2)
plt.plot(test_data.index, exp_smooth_forecast, label='Exp Smoothing Forecast', color='purple', linewidth=2, linestyle='--')
plt.title('Exponential Smoothing Forecast vs Actual', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Close Price (INR)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Phase 7: Compare Baseline Models

In [ ]:
# Create comparison dataframe
comparison_df = pd.DataFrame({
    'ARIMA': arima_metrics,
    'Exponential Smoothing': exp_smooth_metrics
})

print("\n=== Model Comparison ===")
print(comparison_df)

# Visualize comparison
comparison_df.T.plot(kind='bar', figsize=(10, 6), rot=0)
plt.title('Baseline Models Comparison', fontsize=14, fontweight='bold')
plt.ylabel('Error Value')
plt.xlabel('Model')
plt.legend(title='Metric')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Next Steps

### Month 2 Objectives:

1. **LSTM Implementation**
   - Build univariate LSTM for close price prediction
   - Expand to multivariate LSTM with technical indicators
   - Implement sequence-to-sequence architecture
   - Hyperparameter tuning with Optuna

2. **GRU Implementation**
   - Compare GRU vs LSTM performance
   - Analyze training speed differences

3. **Attention Mechanisms**
   - Add attention layer to LSTM/GRU
   - Visualize attention weights
   - Interpret which features the model focuses on

4. **Temporal Fusion Transformer**
   - Implement TFT using pytorch-forecasting
   - Multi-horizon forecasting (1-day, 1-week, 1-month)
   - Quantile predictions for uncertainty estimation

5. **Advanced Evaluation**
   - Walk-forward validation
   - Directional accuracy (did we predict up/down correctly?)
   - Sharpe ratio on trading strategy
   - Maximum drawdown analysis

### Resources:
- PyTorch Forecasting docs: https://pytorch-forecasting.readthedocs.io/
- TFT paper: https://arxiv.org/abs/1912.09363
- Time series cross-validation: https://scikit-learn.org/stable/modules/cross_validation.html#time-series-split

### Code Organization:
```
project1_stock_forecasting/
├── data/
│   ├── raw/              # Downloaded stock data
│   └── processed/        # Feature-engineered datasets
├── notebooks/
│   ├── 01_data_exploration.ipynb
│   ├── 02_baseline_models.ipynb  ← YOU ARE HERE
│   ├── 03_lstm_gru.ipynb         ← NEXT
│   └── 04_tft_advanced.ipynb
├── src/
│   ├── data_pipeline.py
│   ├── feature_engineering.py
│   ├── models/
│   └── evaluation.py
└── README.md
```

## Save Your Work

In [ ]:
# Save processed data
train_data.to_csv('train_data.csv')
test_data.to_csv('test_data.csv')

# Save baseline metrics
comparison_df.to_csv('baseline_metrics.csv')

print("Data saved successfully!")
print("Ready to move to deep learning models in the next notebook.")